# PIPO Data Buffer Simulator (EC2201 Digital Electronics)
**Author:** AI Pair Programmer / Student  
**Course:** EC2201 Digital Systems & Microprocessors  
**Topic:** Parallel-In Parallel-Out (PIPO) Register & Multi-Stage Data Buffer Simulator

---

## 1. Executive Summary & Digital Logic Theory
A **Parallel-In Parallel-Out (PIPO)** data buffer is a digital storage register constructed from $N$ parallel D Flip-Flops sharing a common clock ($CLK$) and control signals. On a rising clock edge:
- **Parallel Input:** Data bits $D_0, D_1, \dots, D_{N-1}$ are sampled simultaneously.
- **Parallel Output:** Data outputs $Q_0, Q_1, \dots, Q_{N-1}$ update concurrently.
- **Latency:** 1 clock cycle for parallel transfer.

```
       D[N-1:0] (Parallel Input Bus)
          │
          ▼
┌──────────────────┐
│  PIPO REGISTER   │◄── CLK (Rising Edge)
│ (N D Flip-Flops) │◄── LOAD (Clock Enable)
└──────────────────┘◄── RST_N (Async Reset)
          │
          ▼
       Q[N-1:0] (Parallel Output Bus)
```


In [ ]:
# Import digital logic simulator modules
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from pipo_register import PIPORegister, DFlipFlop
from buffer_simulator import PIPODataBuffer
from ai_analytics import SyntheticTrafficGenerator, AIFaultDetector
from test_suite import PIPOTestSuite

print("All PIPO simulator modules imported successfully!")


## 2. Core Hardware Simulation: 8-Bit PIPO Register & Truth Table

In [ ]:
# Instantiate 8-bit PIPO Register
pipo = PIPORegister(bit_width=8)

# 1. Present Input Data 0xA5 = 10100101 (Binary)
d_input = [1, 0, 1, 0, 0, 1, 0, 1]

# Clock Low State (CLK=0)
st_low = pipo.clock_step(clk=0, d_bus=d_input, load=1, reset_n=1, oe=1)
print(f"CLK=0 (Low): Q Output = {st_low['q_hex']} | Bus Output = {st_low['q_bus_output']}")

# Clock Rising Edge (CLK=1)
st_high = pipo.clock_step(clk=1, d_bus=d_input, load=1, reset_n=1, oe=1)
print(f"CLK=1 (Rising Edge): Q Output = {st_high['q_hex']} | Bus Output = {st_high['q_bus_output']}")

# Display Hardware Truth Table
print("
--- PIPO Register Truth Table ---")
df_tt = PIPORegister.generate_truth_table(bit_width=8)
df_tt


## 3. Multi-Stage PIPO Buffer Pipeline & Throughput Analysis

In [ ]:
# Instantiate 8-stage PIPO FIFO Buffer operating at 100 MHz clock frequency
buf = PIPODataBuffer(depth=8, bit_width=8, clk_freq_mhz=100.0)

# Write 8 parallel words (Fill pipeline)
burst_words = [0x11, 0x22, 0x33, 0x44, 0x55, 0x66, 0x77, 0x88]
for w in burst_words:
    buf.step_clock(write_enable=1, read_enable=0, write_word=w)

# Read 8 parallel words back out
read_results = []
for _ in range(8):
    rec = buf.step_clock(write_enable=0, read_enable=1)
    read_results.append(rec["read_word"])

print(f"Sent Burst: {[hex(w) for w in burst_words]}")
print(f"Read Burst: {[hex(r) for r in read_results]}")

# Print Global Performance Metrics Summary
summary = buf.get_performance_summary()
pd.DataFrame([summary])


## 4. AI & Data Layer: Fault Detection & Anomaly Classification

In [ ]:
# Generate 200 synthetic workload traffic cycles with 10% fault injection
df_traffic = SyntheticTrafficGenerator.generate_workload(num_samples=200, pattern_type="burst", noise_prob=0.10)

# Train ML Fault Detector (IsolationForest + RandomForest)
detector = AIFaultDetector()
train_metrics = detector.train(df_traffic)

print(f"Total Traffic Samples: {train_metrics['training_samples']}")
print(f"Detected Anomalies: {train_metrics['detected_anomalies']}")

# Feature Importance Plot
plt.figure(figsize=(8, 4))
plt.barh(list(train_metrics['feature_importance'].keys()), list(train_metrics['feature_importance'].values()), color="#38bdf8")
plt.title("AI Fault Detector: Feature Importance Breakdown")
plt.xlabel("Relative Importance Score")
plt.tight_layout()
plt.show()


## 5. Automated Test Suite Execution (10 Normal + 5 Edge/Fault Cases)

In [ ]:
# Run full automated test suite
runner = PIPOTestSuite()
df_test_results = runner.run_all_tests()

pass_count = (df_test_results['Status'] == 'PASS').sum()
print(f"Test Suite Results: {pass_count} / {len(df_test_results)} PASSED")

df_test_results


## 6. Waveform Signal Timing Diagram Plot

In [ ]:
# Plot timing waveforms for PIPO Register operations
cycles = list(range(1, 13))
clk_wave =  [0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1]
load_wave = [1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1]
rst_wave =  [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1]

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(10, 6), sharex=True)

ax1.step(cycles, clk_wave, where='post', color='green', linewidth=2)
ax1.set_ylabel("CLK")
ax1.grid(True, linestyle='--', alpha=0.5)

ax2.step(cycles, load_wave, where='post', color='blue', linewidth=2)
ax2.set_ylabel("LOAD")
ax2.grid(True, linestyle='--', alpha=0.5)

ax3.step(cycles, rst_wave, where='post', color='red', linewidth=2)
ax3.set_ylabel("RST_N")
ax3.set_xlabel("Clock Cycle T")
ax3.grid(True, linestyle='--', alpha=0.5)

plt.suptitle("PIPO Register Digital Signal Timing Waveforms", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
